<a href="https://colab.research.google.com/github/SiddharthKalra2005/ReAct-AI/blob/main/ReAct_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install -q langgraph langchain-google-genai langchain-core

In [31]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "your_api_key_here"

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Quick test
response = llm.invoke("Say ready in one word")
print(response.content)

Ready


In [32]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Useful for math calculations. Input should be a math expression like '2 + 2' or '10 * 5'."""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Error: invalid math expression"

@tool
def word_counter(text: str) -> str:
    """Counts the number of words in a given text."""
    count = len(text.split())
    return f"The text has {count} words."

@tool
def reverse_text(text: str) -> str:
    """Reverses any given text."""
    return text[::-1]

tools = [calculator, word_counter, reverse_text]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['calculator', 'word_counter', 'reverse_text']


In [33]:
from langgraph.prebuilt import create_react_agent

# Create the agent
agent = create_react_agent(llm, tools)

print("LangGraph ReAct Agent ready!")

LangGraph ReAct Agent ready!


/tmp/ipykernel_2395/3986988965.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [34]:
import time
from langchain_core.messages import HumanMessage

# Test 1 - Math
print("=" * 50)
print("TEST 1: Math")
print("=" * 50)
result = agent.invoke({
    "messages": [HumanMessage(content="What is 1234 * 5678?")]
})
print(result["messages"][-1].content)

time.sleep(10)  # wait 10 seconds

# Test 2 - Word count
print("\n" + "=" * 50)
print("TEST 2: Word Counter")
print("=" * 50)
result = agent.invoke({
    "messages": [HumanMessage(content="How many words are in this sentence: The quick brown fox jumps over the lazy dog")]
})
print(result["messages"][-1].content)

time.sleep(10)  # wait 10 seconds

# Test 3 - Multi step
print("\n" + "=" * 50)
print("TEST 3: Multi-step reasoning")
print("=" * 50)
result = agent.invoke({
    "messages": [HumanMessage(content="Reverse the word 'hello' and then count how many characters the result has. Then multiply that by 10.")]
})
print(result["messages"][-1].content)

TEST 1: Math
1234 * 5678 = 7006652

TEST 2: Word Counter
The sentence has 9 words.

TEST 3: Multi-step reasoning
The word 'hello' reversed is 'olleh'. It has 5 characters. 5 multiplied by 10 is 50.


In [35]:
def run_agent(question):
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })
    last = result["messages"][-1].content
    # handle both string and list response formats
    if isinstance(last, list):
        for block in last:
            if isinstance(block, dict) and block.get("type") == "text":
                return block["text"]
    return last

# Demo all 3 cleanly
questions = [
    "What is 999 * 111?",
    "How many words are in: Artificial intelligence is transforming the world",
    "Reverse the word 'LangGraph' and multiply the character count by 7"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {run_agent(q)}")
    print("-" * 40)
    time.sleep(30)

Q: What is 999 * 111?
A: 999 * 111 = 110889
----------------------------------------
Q: How many words are in: Artificial intelligence is transforming the world
A: The text has 6 words.
----------------------------------------
Q: Reverse the word 'LangGraph' and multiply the character count by 7
A: The reversed word is 'hparGgnaL' and the character count multiplied by 7 is 63.
----------------------------------------


In [36]:
# Stream mode - shows every step of reasoning
print("Q: What is 123 * 456 and how many words is that question?")
print("=" * 50)

for step in agent.stream(
    {"messages": [HumanMessage(content="What is 123 * 456 and how many words is that question?")]},
    stream_mode="values"
):
    last_msg = step["messages"][-1]
    last_msg.pretty_print()
    print("-" * 40)

Q: What is 123 * 456 and how many words is that question?
================================ Human Message =================================

What is 123 * 456 and how many words is that question?
----------------------------------------
================================== Ai Message ==================================
Tool Calls:
  calculator (9627350a-23e9-4a47-a3ea-59e5bde16dda)
 Call ID: 9627350a-23e9-4a47-a3ea-59e5bde16dda
  Args:
    expression: 123 * 456
----------------------------------------
================================= Tool Message =================================
Name: calculator

56088
----------------------------------------
================================== Ai Message ==================================
Tool Calls:
  word_counter (360cfeb9-bfce-4cf0-9565-a3aae6952ea4)
 Call ID: 360cfeb9-bfce-4cf0-9565-a3aae6952ea4
  Args:
    text: What is 123 * 456 and how many words is that question?
----------------------------------------
================================= Tool Mess